# Preparasi SNI Instance-Crop v1

Notebook ini **belum melakukan training**. Dua dataset COCO diunduh di server Colab, dinormalisasi menjadi 21 kelas SNI, lalu setiap anotasi diubah menjadi crop klasifikasi. Semua objek dari satu foto sumber tetap berada pada split yang sama. Hasil akhir diarsipkan ke Google Drive agar aman ketika runtime reset.

In [ ]:
# 1/5 — Setup repository, Drive, dan Roboflow
from google.colab import drive, userdata
from pathlib import Path
import json, os, shutil, subprocess, sys

drive.mount('/content/drive')
REPO = Path('/content/coffee-bean-classification')
BRANCH = 'agent/sni-instance-crops'
if not (REPO / '.git').is_dir():
    subprocess.run([
        'git', 'clone', '--branch', BRANCH, '--single-branch',
        'https://github.com/ediprin/coffee-bean-classification.git', str(REPO)
    ], check=True)
else:
    subprocess.run(['git', '-C', str(REPO), 'fetch', 'origin', BRANCH], check=True)
    subprocess.run(['git', '-C', str(REPO), 'checkout', BRANCH], check=True)
    subprocess.run(['git', '-C', str(REPO), 'pull', '--ff-only'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(REPO), 'roboflow'], check=True)

RAW_ROOT = Path('/content/sni-raw')
PREPARED_ROOT = Path('/content/sni-instance-crops')
DRIVE_ROOT = Path('/content/drive/MyDrive/coffee-sni-instance-crop-v1')
ARCHIVE = DRIVE_ROOT / 'sni-instance-crops-v1.tar'
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
print('REPO    :', REPO)
print('OUTPUT  :', PREPARED_ROOT)
print('BACKUP  :', ARCHIVE)

In [ ]:
# 2/5 — Pulihkan hasil dari Drive atau unduh dua dataset ke server Colab
if ARCHIVE.is_file() and not (PREPARED_ROOT / 'audit.json').is_file():
    print('Memulihkan dataset crop dari Google Drive...')
    subprocess.run(['tar', '-xf', str(ARCHIVE), '-C', '/content'], check=True)

if not (PREPARED_ROOT / 'audit.json').is_file():
    from roboflow import Roboflow
    api_key = userdata.get('ROBOFLOW_API_KEY')
    assert api_key, 'Tambahkan Colab Secret bernama ROBOFLOW_API_KEY dan aktifkan akses notebook.'
    rf = Roboflow(api_key=api_key)
    downloads = [
        ('adrian_detection', 'yolo-skripsi-2-lh14g-y61eh', 'coco'),
        ('faruq_segmentation', 'robusta_sni_dataset-hr9ci', 'coco-segmentation'),
    ]
    for folder, project_id, export_format in downloads:
        destination = RAW_ROOT / folder
        if not list(destination.rglob('_annotations.coco.json')):
            print(f'Unduh {folder} ({export_format})...')
            dataset = (rf.workspace('situju-kamkape')
                       .project(project_id).version(1)
                       .download(export_format, location=str(destination)))
            print('DATASET:', dataset.location)
        else:
            print('SKIP raw:', destination)
else:
    print('SKIP download: dataset crop sudah tersedia.')

In [ ]:
# 3/5 — Audit, grouped split, dan crop per objek (CPU; ada progress setiap 250 foto)
if not (PREPARED_ROOT / 'audit.json').is_file():
    subprocess.run([
        sys.executable, '-u', '-m',
        'bilinear_lmmd.data.preparation.prepare_sni_instance_crops',
        '--adrian-root', str(RAW_ROOT / 'adrian_detection'),
        '--faruq-root', str(RAW_ROOT / 'faruq_segmentation'),
        '--output-root', str(PREPARED_ROOT),
        '--seed', '42', '--margin', '0.10'
    ], check=True)
else:
    print('SKIP preparasi: audit lengkap ditemukan.')
assert (PREPARED_ROOT / 'audit.json').is_file()

In [ ]:
# 4/5 — Simpan satu arsip ke Drive (aman dari reset runtime)
if not ARCHIVE.is_file():
    local_archive = Path('/content/sni-instance-crops-v1.tar')
    subprocess.run([
        'tar', '-cf', str(local_archive), '-C', '/content', PREPARED_ROOT.name
    ], check=True)
    shutil.copy2(local_archive, ARCHIVE)
for name in ('audit.json', 'manifest.csv',
             'contact_sheet_adrian_detection.jpg',
             'contact_sheet_faruq_segmentation.jpg'):
    source = PREPARED_ROOT / name
    if source.is_file():
        shutil.copy2(source, DRIVE_ROOT / name)
print('BACKUP SELESAI:', ARCHIVE, f'({ARCHIVE.stat().st_size / 1024**3:.2f} GB)')

In [ ]:
# 5/5 — Ringkasan audit dan sampel visual train
from IPython.display import display
from PIL import Image
audit = json.loads((PREPARED_ROOT / 'audit.json').read_text())
print('Status       :', audit['status'])
print('Kelas        :', audit['num_classes'])
print('Input images :', audit['input_images'])
print('Output crops :', audit['output_crops'])
print('Split total  :', audit['split_totals'])
print('Split ratio  :', {k: f'{v:.2%}' for k, v in audit['split_ratios'].items()})
print('Leak baru    :', audit['generated_cross_split_identity_count'])
print('Duplikat crop dibuang:', audit['removed_exact_duplicate_crops'])
print('Konflik label crop:', audit['conflicting_exact_duplicate_crop_groups'])
print('TEST LOCKED  :', audit['test_locked'])
for relative in audit['contact_sheets']:
    print(relative)
    display(Image.open(PREPARED_ROOT / relative))
print('\nJika contact sheet dan label benar, kirim ringkasan audit di atas. Jangan training sebelum audit visual disetujui.')